# Step 36 — Production Strategy Final\n
## Data Verification

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

FINAL_DIR = Path("../Market_Data/final")
df = pd.read_parquet(FINAL_DIR / "step34_3_corrected.parquet")

ic_check = df.groupby("date", group_keys=False).apply(
    lambda x: x["pred_score"].corr(x["actual_return"], method="spearman"),
    include_groups=False
).mean()
assert ic_check > 0.020, f"Wrong data source: IC={ic_check:.4f}"
print(f"IC verified: {ic_check:.6f}")

IC verified: 0.026841


## P1_Weekly Baseline Reproduction

In [3]:
# Regime filter
daily_ic = df.groupby("date", group_keys=False).apply(
    lambda x: x["pred_score"].corr(x["actual_return"], method="spearman"),
    include_groups=False
)
rolling_ic = daily_ic.shift(1).rolling(20, min_periods=20).mean()
include_regime = rolling_ic > 0.0

df_filtered = df.copy()
valid_dates = include_regime[include_regime].index
df_filtered = df_filtered[df_filtered["date"].isin(valid_dates)].copy()
df_filtered["rank_pct"] = df_filtered.groupby("date")["pred_score"].rank(pct=True)

all_dates = sorted(df_filtered["date"].unique())
all_tickers = sorted(df["ticker"].unique())
all_full_dates = sorted(df["date"].unique())

def simulate_p1_weekly(df_sub, all_tickers_list, cost_bps=10.0):
    dates = sorted(df_sub["date"].unique())
    rebalance_days = set(dates[::5])
    wp = pd.DataFrame(0.0, index=dates, columns=all_tickers_list)
    for i, date in enumerate(dates):
        if date in rebalance_days or i == 0:
            day_data = df_sub[df_sub["date"] == date]
            scores = day_data.set_index("ticker")["pred_score"].clip(lower=0)
            s = scores.sum()
            if s > 0:
                wp.loc[date, scores.index] = scores / s
        else:
            wp.loc[date] = wp.iloc[i - 1]

    ret_pivot = df_sub.pivot(index="date", columns="ticker", values="actual_return").fillna(0)
    common_cols = wp.columns.intersection(ret_pivot.columns)
    wp_a = wp[common_cols]
    ret_a = ret_pivot.reindex(columns=common_cols, fill_value=0)

    td = wp_a.diff().abs().sum(axis=1).fillna(0)
    wtm1 = wp_a.shift(1).fillna(0)
    pr = (wtm1 * ret_a).sum(axis=1)
    nr = pr - td * (cost_bps / 10000)

    avg_turnover = td.mean()
    ann_return = nr.mean() * 252
    net_sharpe = (nr.mean() / (nr.std() + 1e-12)) * np.sqrt(252)
    gross_sharpe = (pr.mean() / (pr.std() + 1e-12)) * np.sqrt(252)
    cum_ret = np.cumprod(1 + nr)
    max_dd = (cum_ret / np.maximum.accumulate(cum_ret) - 1).min()
    eff_n = (1 / (wp_a**2).sum(axis=1).replace(0, float("nan"))).mean()

    prev = wp_a.shift(1).fillna(0)
    curr = wp_a.fillna(0)
    active_today = (curr > 0)
    active_prev = (prev > 0)
    intersection = (active_today & active_prev).sum(axis=1)
    union = (active_today | active_prev).sum(axis=1)
    overlap = (intersection / union.replace(0, float("nan"))).mean()

    breakeven_bps = (pr.mean() / (td.mean() + 1e-12)) * 10000

    downside = nr[nr < 0]
    downside_std = downside.std() * np.sqrt(252) if len(downside) > 0 else 1e-12
    sortino = ann_return / (downside_std + 1e-12)
    calmar = ann_return / (abs(max_dd) + 1e-12)

    return {
        "net_sharpe": net_sharpe, "gross_sharpe": gross_sharpe,
        "turnover": avg_turnover, "ann_return": ann_return,
        "max_dd": max_dd, "effective_n": eff_n,
        "overlap": overlap, "breakeven_bps": breakeven_bps,
        "sortino": sortino, "calmar": calmar,
        "net_returns": nr, "gross_returns": pr,
        "turnover_daily": td, "weight_pivot": wp_a,
    }

# Reproduce P1_Weekly baseline
p1_baseline = simulate_p1_weekly(df_filtered, all_tickers)

BASELINE = {"net_sharpe": 2.9650, "turnover": 0.3062,
            "ann_return": 0.5549, "max_dd": -0.1115, "effective_n": 9.8}
for k, v in BASELINE.items():
    assert abs(p1_baseline[k] - v) < 0.10, \
        f"P1 mismatch {k}: {p1_baseline[k]:.4f} vs {v:.4f}"
print("P1_Weekly baseline confirmed.")

P1_Weekly baseline confirmed.


## Section 1 — Walk-forward validation with robustness gate

In [4]:
w1_dates = all_full_dates[0:82]
w2_dates = all_full_dates[82:163]
w3_dates = all_full_dates[163:245]

windows = {
    "W1_JanApr": w1_dates,
    "W2_MayAug": w2_dates,
    "W3_SepDec": w3_dates,
}

window_results = {}
for name, window_dates in windows.items():
    df_w = df_filtered[df_filtered["date"].isin(window_dates)]
    if len(df_w) == 0:
        print(f"{name}: NO DATA (all filtered by regime)")
        continue
    window_results[name] = simulate_p1_weekly(df_w, all_tickers)
    print(f"{name}: Net Sharpe={window_results[name]['net_sharpe']:.4f}, "
          f"Turnover={window_results[name]['turnover']:.4f}, "
          f"Max DD={window_results[name]['max_dd']:.4f}")

# Robustness gate
positive_sharpe_windows = sum(
    1 for r in window_results.values() if r["net_sharpe"] > 0
)
profitable_windows = sum(
    1 for r in window_results.values() if r["ann_return"] > 0
)
win_rate = positive_sharpe_windows / 3

print(f"\nPositive Sharpe windows: {positive_sharpe_windows}/3")
print(f"Profitable windows:      {profitable_windows}/3")
print(f"Win rate:                {win_rate:.1%}")

assert positive_sharpe_windows >= 2, \
    f"Strategy not robust — only {positive_sharpe_windows}/3 windows positive Sharpe"
assert profitable_windows >= 2, \
    f"Strategy not robust — only {profitable_windows}/3 windows profitable"
print("Robustness gate passed.")

W1_JanApr: Net Sharpe=1.5786, Turnover=0.2480, Max DD=-0.1115
W2_MayAug: Net Sharpe=2.4383, Turnover=0.3209, Max DD=-0.0614
W3_SepDec: Net Sharpe=7.4344, Turnover=0.2938, Max DD=-0.0409

Positive Sharpe windows: 3/3
Profitable windows:      3/3
Win rate:                100.0%
Robustness gate passed.


## Section 2 — Benchmark comparison

In [5]:
# Equal weight all tickers daily
df_eq = df.copy()
n_tickers = df_eq.groupby("date")["ticker"].transform("count")
df_eq["weight"] = 1.0 / n_tickers
wp_eq = df_eq.pivot(index="date", columns="ticker", values="weight").fillna(0)
ret_eq = df_eq.pivot(index="date", columns="ticker", values="actual_return").fillna(0)
td_eq = wp_eq.diff().abs().sum(axis=1).fillna(0)
pr_eq = (wp_eq.shift(1).fillna(0) * ret_eq).sum(axis=1)
nr_eq = pr_eq - td_eq * 10.0 / 10000
eq_sharpe = (nr_eq.mean() / (nr_eq.std() + 1e-12)) * np.sqrt(252)
eq_ann = nr_eq.mean() * 252
eq_dd = (np.cumprod(1 + nr_eq) / np.maximum.accumulate(np.cumprod(1 + nr_eq)) - 1).min()
eq_to = td_eq.mean()

# Buy-hold top 10
df_day1 = df[df["date"] == df["date"].min()].copy()
df_day1["rank_pct"] = df_day1["pred_score"].rank(pct=True)
top10 = df_day1.nlargest(10, "rank_pct")["ticker"].tolist()
wp_bh = pd.DataFrame(0.0, index=sorted(df["date"].unique()), columns=all_tickers)
for t in top10:
    wp_bh.loc[:, t] = 1.0 / len(top10)
ret_bh = df.pivot(index="date", columns="ticker", values="actual_return").fillna(0)
common = wp_bh.columns.intersection(ret_bh.columns)
wp_bh2 = wp_bh[common]
ret_bh2 = ret_bh.reindex(columns=common, fill_value=0)
td_bh = wp_bh2.diff().abs().sum(axis=1).fillna(0)
pr_bh = (wp_bh2.shift(1).fillna(0) * ret_bh2).sum(axis=1)
nr_bh = pr_bh - td_bh * 10.0 / 10000
bh_sharpe = (nr_bh.mean() / (nr_bh.std() + 1e-12)) * np.sqrt(252)
bh_ann = nr_bh.mean() * 252
bh_dd = (np.cumprod(1 + nr_bh) / np.maximum.accumulate(np.cumprod(1 + nr_bh)) - 1).min()
bh_to = td_bh.mean()

# Daily E1_Raw baseline (Step-34.5)
df_e1 = df_filtered.copy()
df_e1["weight"] = df_e1.groupby("date")["pred_score"].transform(
    lambda x: x.clip(lower=0) / (x.clip(lower=0).sum() + 1e-12)
)
wp_e1 = df_e1.pivot(index="date", columns="ticker", values="weight").fillna(0)
ret_e1 = df_e1.pivot(index="date", columns="ticker", values="actual_return").fillna(0)
td_e1 = wp_e1.diff().abs().sum(axis=1).fillna(0)
pr_e1 = (wp_e1.shift(1).fillna(0) * ret_e1).sum(axis=1)
nr_e1 = pr_e1 - td_e1 * 10.0 / 10000
e1_sharpe = (nr_e1.mean() / (nr_e1.std() + 1e-12)) * np.sqrt(252)
e1_ann = nr_e1.mean() * 252
e1_dd = (np.cumprod(1 + nr_e1) / np.maximum.accumulate(np.cumprod(1 + nr_e1)) - 1).min()
e1_to = td_e1.mean()

p1_ann = p1_baseline["ann_return"]

print("| Benchmark | Net Sharpe | Ann Return | Max DD | Turnover | vs P1 Excess Return |")
print("|---|---|---|---|---|---|")
print(f"| P1_Weekly (ours)  | {p1_baseline['net_sharpe']:.4f} | {p1_ann:.4f} | {p1_baseline['max_dd']:.4f} | {p1_baseline['turnover']:.4f} | — |")
print(f"| Equal_weight      | {eq_sharpe:.4f} | {eq_ann:.4f} | {eq_dd:.4f} | {eq_to:.4f} | {p1_ann - eq_ann:+.4f} |")
print(f"| Buy_hold_top10    | {bh_sharpe:.4f} | {bh_ann:.4f} | {bh_dd:.4f} | {bh_to:.4f} | {p1_ann - bh_ann:+.4f} |")
print(f"| Daily_E1_Baseline | {e1_sharpe:.4f} | {e1_ann:.4f} | {e1_dd:.4f} | {e1_to:.4f} | {p1_ann - e1_ann:+.4f} |")

print(f"\n--- Key Finding ---")
print(f"Execution engineering (P1_Weekly vs Daily_E1_Raw):")
print(f"  Net Sharpe improvement: {e1_sharpe:.4f} -> {p1_baseline['net_sharpe']:.4f} (+{p1_baseline['net_sharpe'] - e1_sharpe:.4f})")
print(f"  Turnover reduction:     {e1_to:.4f} -> {p1_baseline['turnover']:.4f} ({(1 - p1_baseline['turnover']/e1_to)*100:.1f}% reduction)")
print(f"  Excess annual return:   {p1_ann - e1_ann:+.4f}")

| Benchmark | Net Sharpe | Ann Return | Max DD | Turnover | vs P1 Excess Return |
|---|---|---|---|---|---|
| P1_Weekly (ours)  | 2.9650 | 0.5549 | -0.1115 | 0.3062 | — |
| Equal_weight      | -0.2253 | -0.0337 | -0.1432 | 1.1437 | +0.5886 |
| Buy_hold_top10    | 1.3775 | 0.1883 | -0.0953 | 0.0000 | +0.3666 |
| Daily_E1_Baseline | 2.1771 | 0.4737 | -0.1324 | 1.2654 | +0.0812 |

--- Key Finding ---
Execution engineering (P1_Weekly vs Daily_E1_Raw):
  Net Sharpe improvement: 2.1771 -> 2.9650 (+0.7879)
  Turnover reduction:     1.2654 -> 0.3062 (75.8% reduction)
  Excess annual return:   +0.0812


## Section 3 — Complete strategy specification

In [6]:
strategy_spec = {
    "signal":              "pred_score (ensemble XGB+LGB+LSTM)",
    "target_horizon":      "t+3 forward return (target_future_return_t3)",
    "universe":            "Nifty 100, 96 tickers",
    "regime_filter":       "dynamic IC-based (rolling 20-day IC > 0, lagged 1 day)",
    "portfolio_side":      "long-only",
    "weight_scheme":       "proportional to clipped pred_score, normalized daily",
    "rebalance_frequency": "weekly (every 5 trading days)",
    "weight_freeze":       "exact freeze on non-rebalance days — no renormalization",
    "cost_model":          "10 bps per unit turnover",
    "effective_n":         "~9.8 positions",
    "data_period":         "2025-01-01 to 2025-12-24",
    "research_lineage":    "Steps 28 -> 33 -> 34.3 -> 34.4 -> 34.5 -> 35 -> 36",
}

print("Complete Strategy Specification:")
print("=" * 60)
for k, v in strategy_spec.items():
    print(f"  {k:25s}: {v}")
print("=" * 60)

Complete Strategy Specification:
  signal                   : pred_score (ensemble XGB+LGB+LSTM)
  target_horizon           : t+3 forward return (target_future_return_t3)
  universe                 : Nifty 100, 96 tickers
  regime_filter            : dynamic IC-based (rolling 20-day IC > 0, lagged 1 day)
  portfolio_side           : long-only
  weight_scheme            : proportional to clipped pred_score, normalized daily
  rebalance_frequency      : weekly (every 5 trading days)
  weight_freeze            : exact freeze on non-rebalance days — no renormalization
  cost_model               : 10 bps per unit turnover
  effective_n              : ~9.8 positions
  data_period              : 2025-01-01 to 2025-12-24
  research_lineage         : Steps 28 -> 33 -> 34.3 -> 34.4 -> 34.5 -> 35 -> 36


## Section 4 — Final performance summary

In [8]:
final = p1_baseline.copy()

print("| Metric              | Value      |")
print("|---|---|")
print(f"| Gross Sharpe        | {final['gross_sharpe']:.4f}     |")
print(f"| Net Sharpe (10bps)  | {final['net_sharpe']:.4f}     |")
print(f"| Annual Return (net) | {final['ann_return']*100:.1f}%     |")
print(f"| Max Drawdown        | {final['max_dd']*100:.1f}%     |")
print(f"| Avg Turnover        | {final['turnover']:.4f}     |")
print(f"| Breakeven Cost      | {final['breakeven_bps']:.1f} bps   |")
print(f"| Effective N         | {final['effective_n']:.1f}        |")
print(f"| Portfolio Overlap   | {final['overlap']*100:.1f}%      |")
print(f"| IC (signal quality) | {ic_check:.4f}     |")
print(f"| Calmar Ratio        | {final['calmar']:.4f}     |")
print(f"| Sortino Ratio       | {final['sortino']:.4f}     |")
print(f"| Win Rate (windows)  | {win_rate:.0%}       |")

# Cost sensitivity
print("\nCost Sensitivity Analysis:")
cost_bps_range = [3, 5, 7, 10, 15, 20]
for cost in cost_bps_range:
    nr = final["gross_returns"] - final["turnover_daily"] * (cost / 10000)
    sr = (nr.mean() / (nr.std() + 1e-12)) * np.sqrt(252)
    print(f"  {cost:2d} bps: Net Sharpe = {sr:.4f}")

print(f"\nBreakeven Cost: {final['breakeven_bps']:.1f} bps")

| Metric              | Value      |
|---|---|
| Gross Sharpe        | 3.3942     |
| Net Sharpe (10bps)  | 2.9650     |
| Annual Return (net) | 55.5%     |
| Max Drawdown        | -11.1%     |
| Avg Turnover        | 0.3062     |
| Breakeven Cost      | 81.9 bps   |
| Effective N         | 9.8        |
| Portfolio Overlap   | 82.6%      |
| IC (signal quality) | 0.0268     |
| Calmar Ratio        | 4.9780     |
| Sortino Ratio       | 4.0923     |
| Win Rate (windows)  | 100%       |

Cost Sensitivity Analysis:
   3 bps: Net Sharpe = 3.2660
   5 bps: Net Sharpe = 3.1803
   7 bps: Net Sharpe = 3.0943
  10 bps: Net Sharpe = 2.9650
  15 bps: Net Sharpe = 2.7491
  20 bps: Net Sharpe = 2.5329

Breakeven Cost: 81.9 bps


## Research Findings Summary\n
\n
1. **Cross-sectional ranking > binary classification**\n
   ML models could not reliably predict Up/Down direction (accuracy ~51%).\n
   Cross-sectional ranking of t+3 forward returns produced IC = 0.0268\n
   and Sharpe > 2.5 gross.\n
\n
2. **Execution engineering > model engineering**\n
   Daily E1_Raw baseline: Net Sharpe 2.18, Turnover 1.265.\n
   P1_Weekly: Net Sharpe 2.97, Turnover 0.306.\n
   A simple weekly freeze produced larger improvement than all model tuning.\n
\n
3. **Target horizon integrity is critical**\n
   Using same-day actual_return instead of target_future_return_t3\n
   collapsed IC from +0.0268 to -0.203 with no change to the signal.\n
   Execution notebooks must preserve the prediction horizon end-to-end.\n
\n
4. **Dynamic regime filtering adds value**\n
   Portfolio E (dynamic IC filter) > Portfolio D (HIGH+MEDIUM) >\n
   Portfolio C (HIGH only) > Portfolio B (all regimes).\n
   Rolling IC regime selection adapts to market conditions.\n
\n
5. **Limitations**\n
   Single calendar year (2025). Indian large-cap universe only.\n
   No market impact modeling. No transaction timing optimization.\n
   Results should be validated on an independent out-of-sample period.\n

In [9]:
# Save results
out_dir = FINAL_DIR / "step36_results"
out_dir.mkdir(parents=True, exist_ok=True)

# Full period metrics
metrics = {k: v for k, v in final.items() if k not in ["net_returns", "gross_returns", "turnover_daily", "weight_pivot"]}
metrics["ic"] = ic_check
metrics["win_rate"] = win_rate
pd.DataFrame([metrics]).to_csv(out_dir / "final_strategy_metrics.csv", index=False)

# Window results
window_export = []
for wname, wr in window_results.items():
    window_export.append({
        "window": wname,
        "net_sharpe": wr["net_sharpe"],
        "turnover": wr["turnover"],
        "ann_return": wr["ann_return"],
        "max_dd": wr["max_dd"],
    })
pd.DataFrame(window_export).to_csv(out_dir / "walkforward_results.csv", index=False)

# Benchmark results
benchmarks_export = [
    {"benchmark": "P1_Weekly", "net_sharpe": p1_baseline["net_sharpe"], "ann_return": p1_baseline["ann_return"], "max_dd": p1_baseline["max_dd"], "turnover": p1_baseline["turnover"]},
    {"benchmark": "Equal_weight", "net_sharpe": eq_sharpe, "ann_return": eq_ann, "max_dd": eq_dd, "turnover": eq_to},
    {"benchmark": "Buy_hold_top10", "net_sharpe": bh_sharpe, "ann_return": bh_ann, "max_dd": bh_dd, "turnover": bh_to},
    {"benchmark": "Daily_E1_Baseline", "net_sharpe": e1_sharpe, "ann_return": e1_ann, "max_dd": e1_dd, "turnover": e1_to},
]
pd.DataFrame(benchmarks_export).to_csv(out_dir / "benchmark_comparison.csv", index=False)

print("=" * 50)
print("RESEARCH PIPELINE COMPLETE")
print("=" * 50)
print(f"Final architecture : P1_Weekly")
print(f"Net Sharpe         : {final['net_sharpe']:.4f}")
print(f"Turnover           : {final['turnover']:.4f}")
print(f"Breakeven cost     : {final['breakeven_bps']:.1f} bps")
print(f"Win rate           : {win_rate:.1%}")
print("=" * 50)
print("\nAll results saved to Market_Data/final/step36_results/")

RESEARCH PIPELINE COMPLETE
Final architecture : P1_Weekly
Net Sharpe         : 2.9650
Turnover           : 0.3062
Breakeven cost     : 81.9 bps
Win rate           : 100.0%

All results saved to Market_Data/final/step36_results/
